In [ ]:
!pip install -q pandas groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import requests
import json
import time
import re

# 1. API Configuration
GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"  # Paste your Groq API Key
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_NAME = "llama-3.1-8b-instant"

# 2. Define the Prompt Template
PROMPT_TEMPLATE = """You are an expert NLP classifier for Nigerian languages and local context text.
Classify the given text into ONE of the following categories:
- Politics
- Sports
- Technology
- Entertainment
- Business
- Religion
- Health

Rules:
1. Respond ONLY with valid JSON.
2. Format: {{"topic": "YOUR_LABEL"}}

Text: {text}
"""

def classify_text(text, api_key):
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}
        ],
        "temperature": 0.0,
        "max_tokens": 100
    }

    try:
        response = requests.post(GROQ_URL, headers=headers, json=payload)
        res_data = response.json()

        if "choices" in res_data:
            content = res_data["choices"][0]["message"]["content"]
            # Extract JSON
            match = re.search(r'\{.*?\}', content, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                return data.get("topic", "Uncategorized")
            return content.strip()
        else:
            return "Error"
    except Exception as e:
        return f"Error: {str(e)}"

# Example Quick Test
sample_text = "The Super Eagles scored two goals in the second half of the AFCON qualifier."
result = classify_text(sample_text, GROQ_API_KEY)
print(f"Sample Classification Result: {result}")

Sample Classification Result: Error


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

# 1. Load data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2. Ultra-Fast TF-IDF Vectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=30000, sublinear_tf=True, min_df=2)
X_train = vectorizer.fit_transform(train_df['text'].fillna(''))
y_train = train_df['category']
X_test = vectorizer.transform(test_df['text'].fillna(''))

# 3. Fast Linear SVM Classifier (Runs in 2 seconds)
clf = LinearSVC(C=1.0, dual='auto')
clf.fit(X_train, y_train)

test_df['predicted_topic'] = clf.predict(X_test)

# 4. Extract Headlines
def extract_headline(text):
    if not isinstance(text, str) or not text.strip():
        return "Iroyin Tuntun"
    clean_text = text.strip()
    first_clause = clean_text.split('.')[0].split('?')[0].split('!')[0].split('\n')[0].strip()
    words = first_clause.split()
    if len(words) > 16:
        return ' '.join(words[:16])
    return first_clause if len(words) >= 3 else clean_text[:80]

test_df['predicted_headline'] = test_df['text'].apply(extract_headline)

# 5. Build Final Submission
submission_rows = []
for idx, row in test_df.iterrows():
    doc_id = row['id']
    submission_rows.append({'id': f"{doc_id}_topic", 'prediction': row['predicted_topic']})
    submission_rows.append({'id': f"{doc_id}_headline", 'prediction': row['predicted_headline']})

submission = pd.DataFrame(submission_rows)
submission.to_csv('submission_fast.csv', index=False)
print("FAST submission created in seconds!")

FAST submission created in seconds!


In [ ]:
# Re-save with UTF-8 BOM so Excel opens it cleanly without garbled characters:
submission.to_csv('submission_fast.csv', index=False, encoding='utf-8-sig')

In [ ]:
import re


def clean_headline(text):
    if not isinstance(text, str):
        return ""

    # Remove extra chatter like "Headline:", "Akukọ:", "Labari:" at the start
    text = re.sub(
        r"^(headline|akuko|labari|title|here is the headline):\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )

    # Remove quotes at the beginning and end
    text = text.strip().strip('"').strip("'")

    # Keep only the very first sentence
    sentences = [s.strip() for s in re.split(r"[.!?\n]", text) if s.strip()]
    if sentences:
        return sentences[0]

    return text

In [ ]:
import pandas as pd

# 1. Load your submission file
df = pd.read_csv("submission.csv")

# 2. Automatically pick the last column (which is where predictions are stored)
target_col = df.columns[-1]
print(f"Cleaning column: '{target_col}'")

# 3. Clean the text in that column
df[target_col] = df[target_col].apply(clean_headline)

# 4. Save the new clean submission file
df.to_csv("submission_cleaned.csv", index=False)
print("Done! 'submission_cleaned.csv' is ready to download and submit to Kaggle.")

Cleaning column: 'prediction'
Done! 'submission_cleaned.csv' is ready to download and submit to Kaggle.


In [13]:
import pandas as pd

# 1. Read your current submission file
df = pd.read_csv("submission_cleaned.csv")

# 2. Keep only rows where 'prediction' is a category label (1-2 words), removing full text rows
clean_df = df[df["prediction"].astype(str).str.split().str.len() <= 2].copy()

# 3. Remove any duplicate IDs, keeping only the first valid prediction
clean_df = clean_df.drop_duplicates(subset=["id"], keep="first")

# 4. Save the final clean file
clean_df.to_csv("submission_final.csv", index=False)

print(f"Success! Created submission_final.csv with {len(clean_df)} clean rows.")
print(clean_df.head(10))

Success! Created submission_final.csv with 1751 clean rows.
                id     prediction
0   yor_0001_topic  entertainment
2   yor_0002_topic  entertainment
4   yor_0003_topic  entertainment
6   yor_0004_topic  entertainment
8   yor_0005_topic  entertainment
10  yor_0006_topic  entertainment
12  yor_0007_topic  entertainment
14  yor_0008_topic  entertainment
16  yor_0009_topic  entertainment
18  yor_0010_topic  entertainment


In [14]:
import pandas as pd

# 1. Load Kaggle's official template (3,486 rows)
sample_sub = pd.read_csv("sample_submission.csv")

# 2. Load your predictions file before it was filtered down
# (Replace 'submission_cleaned.csv' with your output file name if different)
raw_df = pd.read_csv("submission_cleaned.csv")

# 3. Create the final submission matching sample_submission exactly
final_sub = sample_sub.copy()

if len(raw_df) == len(sample_sub):
    final_sub["prediction"] = raw_df["prediction"].values
    print("Successfully mapped all 3,486 predictions!")
else:
    print(f"Notice: raw_df has {len(raw_df)} rows, template has {len(sample_sub)} rows.")

# 4. Save final file
final_sub.to_csv("submission_final.csv", index=False)
print("Saved 'submission_final.csv' successfully!")
print(final_sub.head(10))

Successfully mapped all 3,486 predictions!
Saved 'submission_final.csv' successfully!
                  id                                         prediction
0     yor_0001_topic                                      entertainment
1  yor_0001_headline  Gbajumọ oṣere sinima ni, Nkechi Blessing ni ou...
2     yor_0002_topic                                      entertainment
3  yor_0002_headline  Ogun awitẹlẹ kii pa arọ, eyiun arọ to ba gbọn,...
4     yor_0003_topic                                      entertainment
5  yor_0003_headline  Gbajumọ olorin takansufe nni, Habeb Okikiola, ...
6     yor_0004_topic                                      entertainment
7  yor_0004_headline  Gbajagbaja oṣerebinrin, Funmi Awelewa, ti awọn...
8     yor_0005_topic                                      entertainment
9  yor_0005_headline  Ẹda kii wa laye ko ma ri amiwo nitori ẹni ti y...
